In [1]:
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go

import electricity
import util

In [2]:
utility = electricity.get_utility()
df = pd.json_normalize(utility)
df.columns

Index(['Utility.Number', 'Utility.Name', 'Utility.State', 'Utility.Type',
       'Demand.Summer Peak', 'Demand.Winter Peak', 'Sources.Generation',
       'Sources.Purchased', 'Sources.Other', 'Sources.Total', 'Uses.Retail',
       'Uses.Resale', 'Uses.No Charge', 'Uses.Consumed', 'Uses.Losses',
       'Uses.Total', 'Revenues.Retail', 'Revenue.Delivery', 'Revenue.Resale',
       'Revenue.Adjustments', 'Revenue.Transmission', 'Revenue.Other',
       'Revenue.Total', 'Retail.Residential.Revenue',
       'Retail.Residential.Sales', 'Retail.Residential.Customers',
       'Retail.Commercial.Revenue', 'Retail.Commercial.Sales',
       'Retail.Commercial.Customers', 'Retail.Industrial.Revenue',
       'Retail.Industrial.Sales', 'Retail.Industrial.Customers',
       'Retail.Transportation.Revenue', 'Retail.Transportation.Sales',
       'Retail.Transportation.Customers', 'Retail.Total.Revenue',
       'Retail.Total.Sales', 'Retail.Total.Customers'],
      dtype='str')

In [3]:
# df[["Uses.Retail", "Uses.Resale", "Uses.No Charge", "Uses.Consumed", "Uses.Losses", "Uses.Total"]]

In [4]:
ny_df = util.get_state_data('NY', df)
ny_df = util.prepare_data(ny_df)

ny_df.columns

Index(['Utility.Name', 'Utility.State', 'Utility.Type', 'Sources.Total',
       'Sources.Generation', 'Sources.Purchased', 'Sources.Other',
       'Retail.Residential.Revenue', 'Retail.Residential.Sales',
       'Retail.Residential.Customers', 'Retail.Industrial.Revenue',
       'Retail.Industrial.Sales', 'Retail.Industrial.Customers', 'Uses.Retail',
       'Uses.Losses', 'Uses.Resale', 'Uses.No Charge', 'Uses.Consumed',
       'Uses.Total', 'Demand.Summer Peak', 'Revenues.Retail',
       'ResidentialUnitPrice', 'IndustrialUnitPrice', 'IndustrialRevenueRatio',
       'PriceSpread', 'SystemLossPercentage', 'LoadFactor', 'FairnessIndex'],
      dtype='str')

In [ ]:
corr_matrix = ny_df[['SystemLossPercentage', 'LoadFactor', 'ResidentialUnitPrice', 'IndustrialUnitPrice', 'FairnessIndex']].corr()

fig3 = px.imshow(
    corr_matrix, 
    text_auto='.2f', 
    color_continuous_scale='RdBu_r',
    aspect="auto",
    title='<b>Statistical Significance:</b> Correlation Heatmap of Key Metrics',
    labels=dict(color="Correlation Score"),
    template='plotly_white'
)

fig3.show()

In [ ]:
def get_fairness_scatter(df):
    fig = px.scatter(
        df, x='SystemLossPercentage', y='ResidentialUnitPrice',
        # size='Retail.Residential.Customers', 
        color='Utility.Type',
        hover_name='Utility.Name', 
                        trendline="ols", trendline_scope="overall", trendline_color_override="black",
        title='<b>Fairness Audit:</b> Correlation Between System Loss and Residential Price',
        labels={'Utility.Type': "Ownership Model",
                'SystemLossPercentage': 'System Energy Loss (%)', 
                'ResidentialUnitPrice': 'Residential Price ($/MWh)'},
        template='plotly_white',
        color_discrete_sequence=px.colors.qualitative.Prism
    )

    fig.show()

get_fairness_scatter(util.get_residential_sys_loss(ny_df))

In [11]:
def plot_scatterplot(df):
    fig = px.scatter(df, x='LoadFactor', y='ResidentialUnitPrice', color="Utility.Type",
                    title="",
                    labels={
                            "Utility.Type": "Ownership Model",
                            "LoadFactor": "System Efficiency (Load Factor)",
                            "ResidentialUnitPrice": "Residential Rate ($/MWh)"
                        },
                    trendline="ols", trendline_scope="overall", trendline_color_override="black",
                    template='plotly_white',
                    hover_name="Utility.Name")
    fig.show()

# Remove rows without Residential services)
plot_scatterplot(util.get_residential_load_factor(ny_df))

AttributeError: module 'util' has no attribute 'get_residential_load_factor'

In [8]:
def plot_dumbbell(df):
    # Sort by spread to show the most "unfair" utilities at the top
    df_sorted = df[df.PriceSpread > 0].sort_values('PriceSpread', ascending=True).tail(10)

    fig = go.Figure()

    # 1. Add the lines connecting the dots
    for i, row in df_sorted.iterrows():
        fig.add_shape(
            type='line', x0=row['IndustrialUnitPrice'], x1=row['ResidentialUnitPrice'],
            y0=row['Utility.Name'], y1=row['Utility.Name'],
            line=dict(color='lightgrey', width=2)
        )

    # 2. Add Industrial dots
    fig.add_trace(go.Scatter(
        x=df_sorted['IndustrialUnitPrice'], y=df_sorted['Utility.Name'],
        mode='markers', name='Industrial Rate', marker=dict(color='#1f77b4', size=10)
    ))

    # 3. Add Residential dots
    fig.add_trace(go.Scatter(
        x=df_sorted['ResidentialUnitPrice'], y=df_sorted['Utility.Name'],
        mode='markers', name='Residential Rate', marker=dict(color='#d62728', size=10)
    ))

    fig.update_layout(title="<b>Rate Disparity</b>", 
                      xaxis_title="Rate ($/MWh)", yaxis_title="")
    return fig

# Remove utilities offering ONLY either industrial or residential rates
plot_dumbbell(util.get_customer_utilities(ny_df, "Both"))

In [9]:
def get_state_box_plot(df):
    # Compare Residential Price across Utility Types
    fig = px.box(df, x='Utility.Type', y='ResidentialUnitPrice', color="Utility.Type", 
                 points="all", hover_name="Utility.Name", hover_data=["Retail.Residential.Customers", "LoadFactor"],
                 title="<b>Utility Type:</b> Residential Price Distribution",
                 labels={"Utility.Type": "Ownership Model", "ResidentialUnitPrice":"Residential Rate", 
                         "Retail.Residential.Customers": "Customer Count", "LoadFactor": "Load Factor"},
                 template='plotly_white')
    
    fig.update_layout(
        xaxis_title="", 
        yaxis_title="Residential Rate ($/MWh)"
    )

    fig.show()

# Limit to utilities with retail customers
get_state_box_plot(util.get_customer_utilities(ny_df, "Residential"))

In [ ]:
def plot_sankey(row):
    # 'row' is a single utility's data
    labels = ["Generated", "Purchased", "Other", "Uses", "Retail Sales", 
              "Resale", "Losses", "Consumed", "No Charge"]

    fig = go.Figure(data=[go.Sankey(
        valueformat = ".1f",
        valuesuffix = "%",
        node = dict(
          pad = 15, thickness = 20,
          line = dict(color = "black", width = 0.5),
          label = labels,
          color="blue"
        ),
        link = dict(
          source = [0, 1, 2, 3, 3, 3, 3, 3, 3], 
          target = [3, 3, 3, 4, 5, 6, 7, 8, 9], 
          value = [
              row["Sources.Generation"],
              row["Sources.Purchased"],
              row["Sources.Other"],
              row["Uses.Retail"],
              row["Uses.Resale"],
              row["Uses.Consumed"],
              row["Uses.Losses"],
              row["Uses.No Charge"]
          ],
      hovercolor="yellow"
      ))])

    fig.update_layout(
      title_text=f"<b>Energy Flow: </b>{row['Utility.Name']}",
      hovermode = 'x'
    )

    return fig

plot_sankey(util.get_utility_usage(ny_df.iloc[10]))